# Legal RAG: MiniLM versus Qwen input windows

**Question:** Does a longer-context embedding model recover evidence missed by a short encoder window, or does additional text dilute the relevant signal?

This notebook separates data validation, model execution, development selection, and held-out evaluation. Empty result tables mean the experiment has not been run; no illustrative retrieval scores are substituted.


## 1. Setup
Open this notebook from the `qwen_embeddings` folder. For Colab, upload/extract the folder or clone the team repository, then change into this folder. Use a GPU runtime for Qwen when available. Install requirements in the active environment before enabling execution. A fresh environment avoids modifying unrelated projects.


In [1]:
# Uncomment in a fresh notebook environment when dependencies are needed.
# %pip install -r requirements.txt

In [2]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'experiment.py').exists() and (ROOT / 'qwen_embeddings').exists():
    ROOT = ROOT / 'qwen_embeddings'
assert (ROOT / 'experiment.py').exists(), 'Change directory to qwen_embeddings first.'
sys.path.insert(0, str(ROOT))
from compact_results import restore_records
restore_records()
import experiment
config = experiment.read_json(ROOT / 'config.json')
def run(*args):
    subprocess.run([sys.executable, '-u', str(ROOT / 'run_experiment.py'), *args], cwd=ROOT, check=True)
display(pd.DataFrame(config['models']).T)
print('Qwen document windows:', config['qwen_windows'])
print('Common returned-body budget:', config['context_budget'], 'MiniLM tokens')

,id,revision,query_window
minilm,sentence-transformers/all-MiniLM-L6-v2,1110a243fdf4706b3f48f1d95db1a4f5529b4d41,256
qwen,Qwen/Qwen3-Embedding-0.6B,97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3,512


Qwen document windows: [512, 2048, 4096, 8192]
Common returned-body budget: 2048 MiniLM tokens


## 2. Shared data and evaluation
Children and parents are unchanged from the hierarchy package. Questions retain original gold-child labels and mapped parents. The 20-dev/80-test split is fixed. Three parents occur across both splits, so this is not parent-disjoint evaluation. No answers or labels enter model inputs.


In [3]:
run('audit')
display(pd.Series(experiment.read_json(ROOT / 'results/data_audit.json')['counts'], name='Count').to_frame())
display(pd.Series(experiment.read_json(ROOT / 'results/data_audit.json')['splits'], name='Questions').to_frame())

,Count
children,4876
parents,2785
queries,100


,Questions
dev,20
test,80


## 3. Experimental design
MiniLM uses its 256-token window. Qwen uses 512/2048/4096/8192 document tokens; Qwen query text, instruction, and 512-token query window stay fixed. Both models use their own tokenizer, normalized embeddings and exact cosine search. Their architectures, training and pooling differ, so a model comparison does not isolate context length. The within-Qwen sweep is the window experiment.

We reuse the same 2,048-token **returned-context** budget and tokenizer across every run. It is distinct from the embedding input window. Parent-unit Hit@k is reported separately from inclusion/coverage of the original supporting child after this budget.


In [4]:
grid = [('minilm', corpus, 256) for corpus in ['child', 'parent']]
grid += [('qwen', corpus, w) for corpus in ['child', 'parent'] for w in config['qwen_windows']]
display(pd.DataFrame(grid, columns=['Model', 'Corpus', 'Document window']))

,Model,Corpus,Document window
0,minilm,child,256
1,minilm,parent,256
2,qwen,child,512
3,qwen,child,2048
4,qwen,child,4096
5,qwen,child,8192
6,qwen,parent,512
7,qwen,parent,2048
8,qwen,parent,4096
9,qwen,parent,8192


## 4. Model execution checks
These checks encode two synthetic documents. Passing confirms a working model path, not benchmark quality. Model downloads require network access and disk space.


In [5]:
RUN_SMOKE = False
if RUN_SMOKE:
    for model in ['minilm', 'qwen']:
        run('smoke', '--model', model)
for model in ['minilm', 'qwen']:
    path = ROOT / f'results/smoke_{model}.json'
    print(model, experiment.read_json(path) if path.exists() else 'Not executed')

minilm {'shape': [2, 384], 'ranking': [0, 1], 'hardware': {'platform': 'Windows-11-10.0.26200-SP0', 'processor': 'Intel64 Family 6 Model 204 Stepping 2, GenuineIntel', 'device': 'cuda', 'gpu': 'NVIDIA GeForce RTX 5070 Laptop GPU'}, 'batch_padding_consistency_passed': True, 'scope': 'Two synthetic documents; execution check only, not benchmark results'}
qwen {'shape': [2, 1024], 'ranking': [0, 1], 'hardware': {'platform': 'Windows-11-10.0.26200-SP0', 'processor': 'Intel64 Family 6 Model 204 Stepping 2, GenuineIntel', 'device': 'cuda', 'gpu': 'NVIDIA GeForce RTX 5070 Laptop GPU'}, 'batch_padding_consistency_passed': True, 'scope': 'Two synthetic documents; execution check only, not benchmark results'}


## 5. Development sweep
Enable deliberately: all passages are embedded for each configuration, while only development questions are evaluated. Batch caches allow resuming interruptions. Long windows are resource intensive; do not silently reduce a window after an out-of-memory error.


In [6]:
RUN_DEV = False
if RUN_DEV:
    for model, corpus, window in grid:
        run('dev', '--model', model, '--corpus', corpus, '--window', str(window), '--device', 'cuda', '--batch-size', '16' if model == 'minilm' else '4')
run('report')
table_path = ROOT / 'results/comparison.csv'
if table_path.exists():
    table = pd.read_csv(table_path)
    display(table.loc[(table.split == 'dev') & (table.k.isin([5, 10])),
        ['run_id','k','unit_hit','gold_child_coverage_in_context','gold_body_token_coverage',
         'returned_content_tokens','truncation_rate','latency_mean_ms']])
else:
    print('No completed benchmark runs yet.')

,run_id,k,unit_hit,gold_child_coverage_in_context,gold_body_token_coverage,returned_content_tokens,truncation_rate,latency_mean_ms
1,minilm_child_w256,5,0.30,0.30,0.300000,1552.85,0.550451,6.918900
2,minilm_child_w256,10,0.35,0.35,0.350000,2044.30,0.550451,6.918900
4,minilm_parent_w256,5,0.30,0.10,0.112534,1791.80,0.558707,6.038733
5,minilm_parent_w256,10,0.35,0.10,0.112534,2044.30,0.558707,6.038733
7,qwen_child_w2048,5,0.40,0.40,0.400000,1453.75,0.000000,73.523342
8,qwen_child_w2048,10,0.50,0.50,0.500000,1976.45,0.000000,73.523342
10,qwen_child_w4096,5,0.40,0.40,0.400000,1453.75,0.000000,68.972065
11,qwen_child_w4096,10,0.50,0.50,0.500000,1976.45,0.000000,68.972065
13,qwen_child_w512,5,0.40,0.40,0.400000,1441.85,0.108080,76.734330
14,qwen_child_w512,10,0.50,0.50,0.500000,1976.45,0.108080,76.734330


## 6. Freeze settings, then evaluate test
Choose one Qwen window per corpus by dev gold-body coverage@5; ties use unit Hit@5 and then shorter windows. The test comparison includes MiniLM, Qwen-512, and the selected window. Do not select using test results. Selection requires the entire registered development grid and freezes data/config/code fingerprints.


In [7]:
FREEZE_SELECTION = False
RUN_TEST = False
if FREEZE_SELECTION:
    run('select')
if RUN_TEST:
    subprocess.run([sys.executable, str(ROOT / 'run_gpu.py'), '--test-only'], cwd=ROOT, check=True)
selection_path = ROOT / 'results/selection.json'
if selection_path.exists():
    display(pd.DataFrame(experiment.read_json(selection_path)['runs']))
else:
    print('Settings have not been selected; test has not been unlocked.')

,model,corpus,window
0,minilm,child,256
1,qwen,child,512
2,minilm,parent,256
3,qwen,parent,512


## 7. Retrieval quality, truncation and cost
Hit and recall use ranked-unit labels; body coverage uses the original gold passage and common context budget. Neither measures answer correctness. Corpus encoding, index assembly and query latency are separate. Query latency includes query encoding and exact search, excludes context construction, and uses warmup plus repeated measurements. Hardware must match for direct timing comparisons.


In [8]:
run('report')
if table_path.exists():
    table = pd.read_csv(table_path)
    display(table[table.split == 'test'])
    display(table[['split','run_id','truncation_rate','corpus_encode_seconds',
                   'index_assembly_seconds','latency_mean_ms','latency_p95_ms']].drop_duplicates())
else:
    print('No model-performance conclusions are available.')

,split,run_id,model,corpus,window,mode,k,token_budget,n_queries,ranked_unit_hit_at_k,...,unit_precision,unit_mrr,unit_ndcg,device,gpu,truncation_rate,corpus_encode_seconds,index_assembly_seconds,latency_mean_ms,latency_p95_ms
30,test,minilm_child_w256,minilm,child,256,child,1,2048,80,0.0625,...,0.06250,0.062500,0.062500,cuda,NVIDIA GeForce RTX 5070 Laptop GPU,0.550451,5.422619,0.005964,12.381991,17.911610
31,test,minilm_child_w256,minilm,child,256,child,5,2048,80,0.2875,...,0.05750,0.154583,0.188122,cuda,NVIDIA GeForce RTX 5070 Laptop GPU,0.550451,5.422619,0.005964,12.381991,17.911610
32,test,minilm_child_w256,minilm,child,256,child,10,2048,80,0.3500,...,0.03500,0.162669,0.208105,cuda,NVIDIA GeForce RTX 5070 Laptop GPU,0.550451,5.422619,0.005964,12.381991,17.911610
33,test,minilm_parent_w256,minilm,parent,256,parent,1,2048,80,0.1375,...,0.13750,0.137500,0.137500,cuda,NVIDIA GeForce RTX 5070 Laptop GPU,0.558707,4.438036,0.002278,14.559849,21.529790
34,test,minilm_parent_w256,minilm,parent,256,parent,5,2048,80,0.3375,...,0.06750,0.201458,0.234694,cuda,NVIDIA GeForce RTX 5070 Laptop GPU,0.558707,4.438036,0.002278,14.559849,21.529790
35,test,minilm_parent_w256,minilm,parent,256,parent,10,2048,80,0.4625,...,0.04625,0.219350,0.276324,cuda,NVIDIA GeForce RTX 5070 Laptop GPU,0.558707,4.438036,0.002278,14.559849,21.529790
36,test,qwen_child_w512,qwen,child,512,child,1,2048,80,0.2625,...,0.26250,0.262500,0.262500,cuda,NVIDIA GeForce RTX 5070 Laptop GPU,0.108080,156.389218,0.012377,89.719668,127.223760
37,test,qwen_child_w512,qwen,child,512,child,5,2048,80,0.4625,...,0.09250,0.347083,0.376260,cuda,NVIDIA GeForce RTX 5070 Laptop GPU,0.108080,156.389218,0.012377,89.719668,127.223760
38,test,qwen_child_w512,qwen,child,512,child,10,2048,80,0.5750,...,0.05750,0.361662,0.412194,cuda,NVIDIA GeForce RTX 5070 Laptop GPU,0.108080,156.389218,0.012377,89.719668,127.223760
39,test,qwen_parent_w512,qwen,parent,512,parent,1,2048,80,0.2250,...,0.22500,0.225000,0.225000,cuda,NVIDIA GeForce RTX 5070 Laptop GPU,0.293357,94.547018,0.005993,96.932461,132.011415


,split,run_id,truncation_rate,corpus_encode_seconds,index_assembly_seconds,latency_mean_ms,latency_p95_ms
0,dev,minilm_child_w256,0.550451,5.422619,0.001998,6.918900,9.766160
3,dev,minilm_parent_w256,0.558707,4.438036,0.001059,6.038733,8.751840
6,dev,qwen_child_w2048,0.000000,163.943272,0.016072,73.523342,136.592730
9,dev,qwen_child_w4096,0.000000,165.799206,0.010240,68.972065,98.222915
12,dev,qwen_child_w512,0.108080,156.389218,0.009380,76.734330,98.725855
15,dev,qwen_child_w8192,0.000000,164.252074,0.009750,89.555657,152.343775
18,dev,qwen_parent_w2048,0.039138,231.423498,0.005537,64.359072,69.605690
21,dev,qwen_parent_w4096,0.008259,297.327645,0.004639,78.790787,104.226165
24,dev,qwen_parent_w512,0.293357,94.547018,0.004401,67.056512,92.292915
27,dev,qwen_parent_w8192,0.000718,338.016985,0.004515,66.281925,84.686885


## 8. Where longer windows help or hurt
Compare each longer Qwen window with Qwen-512 on the same questions. Differences below concern gold-body coverage@5. Inspect source passages before interpreting the cause; a score difference does not prove that truncation was the only mechanism.


In [9]:
examples = experiment.read_json(ROOT / 'results/paired_examples.json')
if examples:
    examples = pd.DataFrame(examples)
    display(examples.sort_values('coverage_delta_vs_qwen512', ascending=False).head(5))
    display(examples.sort_values('coverage_delta_vs_qwen512').head(5))
    test_cases = examples[examples.split == 'test']
    children_by_id = {r['id']: r for r in experiment.load_data()['children']}
    for effect in ['helped', 'hurt']:
        matches = test_cases[test_cases.effect == effect]
        if not matches.empty:
            case = matches.iloc[0]
            print('\nHeld-out example:', effect, '|', case.run_id)
            print('Question:', case.question)
            print('Coverage change:', round(case.coverage_delta_vs_qwen512, 4))
            print('Gold passage preview:', children_by_id[case.gold_child_ids[0]]['text'][:600])
else:
    print('No paired differences available yet (runs may be incomplete or rankings unchanged).')

,split,run_id,query_id,question,gold_child_ids,coverage_delta_vs_qwen512,effect,short_ranking,long_ranking
0,dev,qwen_parent_w4096,94,Josh deliberately and dishonestly destroys phy...,[7.5.18-c2-s1],-1.0,hurt,"[7.5.18.1-c2, 7.5.22.11-c2, 7.5.13-c4, 7.5.18-...","[7.5.18.1-c2, 7.5.22.11-c2, 7.5.13-c4, 7.5.12-..."
1,dev,qwen_parent_w8192,94,Josh deliberately and dishonestly destroys phy...,[7.5.18-c2-s1],-1.0,hurt,"[7.5.18.1-c2, 7.5.22.11-c2, 7.5.13-c4, 7.5.18-...","[7.5.18.1-c2, 7.5.22.11-c2, 7.5.13-c4, 7.5.12-..."


,split,run_id,query_id,question,gold_child_ids,coverage_delta_vs_qwen512,effect,short_ranking,long_ranking
0,dev,qwen_parent_w4096,94,Josh deliberately and dishonestly destroys phy...,[7.5.18-c2-s1],-1.0,hurt,"[7.5.18.1-c2, 7.5.22.11-c2, 7.5.13-c4, 7.5.18-...","[7.5.18.1-c2, 7.5.22.11-c2, 7.5.13-c4, 7.5.12-..."
1,dev,qwen_parent_w8192,94,Josh deliberately and dishonestly destroys phy...,[7.5.18-c2-s1],-1.0,hurt,"[7.5.18.1-c2, 7.5.22.11-c2, 7.5.13-c4, 7.5.18-...","[7.5.18.1-c2, 7.5.22.11-c2, 7.5.13-c4, 7.5.12-..."


## 9. Reporting boundary
Report completed configurations and hardware, including failures. Do not describe smoke tests or the prior hierarchy diagnostics as embedding results. Longer windows may change few child inputs; inspect truncation rates. Retain the complete test comparison, and do not claim evidence-body coverage proves answer completeness.

See README.md for command-line use and RESULTS.md for the delivered validation status.


## 10. Meeting summary: held-out results
This table is populated only after the frozen test runs finish. Compare model changes separately from within-Qwen window changes. Evidence coverage is measured after the common returned-context budget; it is not answer correctness. Read RESULTS.md for the interpretation and run status.


In [10]:
if table_path.exists():
    final = pd.read_csv(table_path)
    heldout = final[final.split == 'test']
    if not heldout.empty:
        compact = heldout[heldout.k == 5][['model','corpus','window','unit_hit',
            'gold_child_coverage_in_context','gold_body_token_coverage','latency_mean_ms']].copy()
        compact.columns = ['Model','Corpus','Input window','Hit@5','Full evidence@5',
                           'Gold-token coverage@5','Query latency (ms)']
        display(compact.round(4))
        print('Each row uses 80 held-out questions. Window selection used development questions only.')
    else:
        print('Held-out evaluation has not completed.')


,Model,Corpus,Input window,Hit@5,Full evidence@5,Gold-token coverage@5,Query latency (ms)
31,minilm,child,256,0.2875,0.2875,0.2875,12.3820
34,minilm,parent,256,0.3375,0.2500,0.2680,14.5598
37,qwen,child,512,0.4625,0.4500,0.4617,89.7197
40,qwen,parent,512,0.4750,0.3875,0.4043,96.9325


Each row uses 80 held-out questions. Window selection used development questions only.
